# Tagging & Data Classification Maturity Audit Notebook

Checks whether Snowflake's native object tagging and sensitive-data classification features
are actually in use in this account — and if they are, this notebook (and its results) should
largely replace the name-matching heuristics used in the security posture, sharing, access
audit, and data lifecycle notebooks, since real classification is a much stronger source of
truth than pattern-matching on column names. Run top to bottom. It surfaces:

1. **Tag inventory** — every tag defined, and how many objects/columns actually carry one
2. **Coverage by database/schema** — where tagging has and hasn't been adopted
3. **Sensitivity-relevant tags** — tags whose name or values suggest they're used for data classification (PII, PHI, CONFIDENTIAL, etc.)
4. **Auto-classification results** — Snowflake's built-in classification feature, if it has ever been run
5. **Heuristic vs. real classification gap check** — cross-references the name-matching results from earlier notebooks against actual classification tags
6. **Tag-based masking policy association** — whether tags are wired to policies, or exist as labels only
7. **A maturity scorecard**

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on `SNOWFLAKE` (or `ACCOUNTADMIN`).
- Object tagging and classification are Enterprise-edition-or-above features — if Sections 1-2
  come back essentially empty, that's the finding: not adopted yet, not "nothing to see here."
- Nothing here is destructive. Every cell reads.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET classification_tag_name_patterns = '%PII%,%PHI%,%CONFIDENTIAL%,%CLASSIFICATION%,%SENSITIV%,%DATA_CLASS%,%PRIVACY%';
SET sensitive_column_patterns = 'SSN,SOCIAL_SEC,PATIENT,MRN,DOB,BIRTH,DIAGNOS,EMAIL,PHONE,ADDRESS,INSURANCE,MEDICAL,HEALTH,GENDER,RACE,ZIP,CREDIT_CARD';

SELECT $classification_tag_name_patterns AS classification_tag_name_patterns,
       $sensitive_column_patterns AS sensitive_column_patterns;


## 1. Tag inventory

Every tag defined in the account. An account with zero or a handful of tags, none of them
applied anywhere, means classification is not yet a real capability here — worth knowing
before assuming any earlier notebook's masking-coverage findings can be cross-checked against
"official" classification.


In [ ]:
-- All tags defined
SELECT
    tag_database,
    tag_schema,
    tag_name,
    created,
    comment
FROM snowflake.account_usage.tags
WHERE deleted IS NULL
ORDER BY tag_database, tag_schema, tag_name;


In [ ]:
-- Actual tag applications (object/column level) — this is where you see whether tags are
-- just defined or actually adopted
SELECT
    tag_database,
    tag_schema,
    tag_name,
    tag_value,
    object_database,
    object_schema,
    object_name,
    column_name,
    domain
FROM snowflake.account_usage.tag_references
ORDER BY tag_database, tag_schema, tag_name, object_database, object_schema
LIMIT 500;


## 2. Coverage by database/schema

How much of the account is actually tagged, broken down so you can see whether adoption is
account-wide, limited to one team's pilot, or effectively zero.


In [ ]:
-- Tag application count and distinct tagged objects, by database
SELECT
    object_database,
    COUNT(DISTINCT tag_name)                                      AS distinct_tags_used,
    COUNT(DISTINCT object_database || '.' || object_schema || '.' || object_name) AS distinct_tagged_objects,
    COUNT(*)                                                       AS total_tag_applications
FROM snowflake.account_usage.tag_references
GROUP BY object_database
ORDER BY total_tag_applications DESC;


In [ ]:
-- Total table/column universe, for comparison against tag coverage above
SELECT
    COUNT(DISTINCT table_catalog || '.' || table_schema || '.' || table_name) AS total_tables,
    COUNT(*) AS total_columns
FROM snowflake.account_usage.columns
WHERE deleted IS NULL AND table_schema != 'INFORMATION_SCHEMA';


## 3. Sensitivity-relevant tags

Tags whose name suggests they're used for data classification specifically (as opposed to,
say, cost-allocation or environment tags like `COST_CENTER` or `ENV`), and what values are
actually in use for them.


In [ ]:
-- Tags that look classification-related by name
SELECT DISTINCT
    tag_database, tag_schema, tag_name
FROM snowflake.account_usage.tags
WHERE deleted IS NULL
  AND EXISTS (
        SELECT 1 FROM TABLE(SPLIT_TO_TABLE($classification_tag_name_patterns, ',')) p
        WHERE tag_name ILIKE p.value
      )
ORDER BY tag_name;


In [ ]:
-- Distinct values in use for those tags, and how many objects carry each value
SELECT
    tr.tag_name,
    tr.tag_value,
    COUNT(*) AS application_count
FROM snowflake.account_usage.tag_references tr
WHERE EXISTS (
        SELECT 1 FROM TABLE(SPLIT_TO_TABLE($classification_tag_name_patterns, ',')) p
        WHERE tr.tag_name ILIKE p.value
      )
GROUP BY tr.tag_name, tr.tag_value
ORDER BY tr.tag_name, application_count DESC;


## 4. Auto-classification results

Snowflake's built-in sensitive-data classification, if it has been run, applies system tags
(`SNOWFLAKE.CORE.SEMANTIC_CATEGORY` and `SNOWFLAKE.CORE.PRIVACY_CATEGORY`) directly to
columns. This is the strongest available signal — it's Snowflake's own classification engine,
not a name-matching guess.


In [ ]:
-- Any classification profiles configured
SHOW CLASSIFICATION PROFILES;


In [ ]:
-- Columns carrying Snowflake's auto-classification system tags, if any classification has run
SELECT
    tag_name,
    tag_value,
    object_database,
    object_schema,
    object_name,
    column_name
FROM snowflake.account_usage.tag_references
WHERE tag_database = 'SNOWFLAKE'
  AND tag_schema = 'CORE'
  AND tag_name IN ('SEMANTIC_CATEGORY', 'PRIVACY_CATEGORY')
ORDER BY object_database, object_schema, object_name, column_name;


If Section 4 returns nothing, classification has likely never been run in this account —
that's a concrete, high-value early recommendation: running classification against the
databases flagged as sensitive by the earlier heuristic-based notebooks would replace guesswork
with Snowflake's actual semantic detection.


## 5. Heuristic vs. real classification gap check

Cross-references the column-name heuristic used throughout the earlier notebooks against
whatever real classification data exists (tag-based, Section 1-3, or auto-classification,
Section 4). Two useful outputs: columns the heuristic *missed* that are actually classified as
sensitive, and columns the heuristic *flagged* that classification does not confirm (possible
false positive, or simply not yet classified).


In [ ]:
-- Columns with real classification/sensitivity tags that DON'T match the naming heuristic
-- (things the name-matching approach in earlier notebooks would have missed entirely)
WITH classified AS (
    SELECT DISTINCT object_database, object_schema, object_name, column_name, tag_name, tag_value
    FROM snowflake.account_usage.tag_references
    WHERE (tag_database = 'SNOWFLAKE' AND tag_schema = 'CORE' AND tag_name IN ('SEMANTIC_CATEGORY', 'PRIVACY_CATEGORY'))
       OR EXISTS (SELECT 1 FROM TABLE(SPLIT_TO_TABLE($classification_tag_name_patterns, ',')) p WHERE tag_name ILIKE p.value)
)
SELECT
    object_database, object_schema, object_name, column_name, tag_name, tag_value
FROM classified c
WHERE NOT EXISTS (
        SELECT 1 FROM TABLE(SPLIT_TO_TABLE($sensitive_column_patterns, ',')) p
        WHERE c.column_name ILIKE '%' || TRIM(p.value) || '%'
      )
ORDER BY object_database, object_schema, object_name;


## 6. Tag-based masking policy association

Tags can drive masking policy assignment automatically (tag an column once, and any masking
policy associated with that tag applies) — a much more scalable model than assigning masking
policies object-by-object. This checks whether that association is actually configured.


In [ ]:
-- Masking policies and the tags they're associated with, where configured
SHOW MASKING POLICIES;


For each policy above, run `DESCRIBE MASKING POLICY <name>;` and separately check
`ALTER TAG <tag_name> SET MASKING POLICY <policy_name>` history in query history (or simply
ask the team directly) to confirm whether any masking policy is tag-driven versus applied
object-by-object. There isn't a single clean account-wide view for this association across all
Snowflake releases, so this is the one check in this notebook worth doing manually alongside
the query results.


## 7. Maturity scorecard

Not a query — a qualitative rollup from the sections above.

| Dimension | Finding | Maturity |
|---|---|---|
| Tags defined at all | *Section 1* | None / Ad hoc / Standardized |
| Tag coverage vs. total object count | *Section 2* | None / Pilot / Broad |
| Dedicated classification tags in use | *Section 3* | None / Informal / Governed |
| Snowflake auto-classification run | *Section 4* | Never / Run once / Scheduled |
| Tag-driven masking configured | *Section 6* | None / Partial / Comprehensive |


### Notes, caveats, and next steps

- **If this account has real classification, prefer it everywhere.** The security posture,
  sharing/listings, access audit trail, and data lifecycle notebooks all fall back to
  column-name matching for sensitivity detection. Once this notebook shows real tags exist,
  swap those notebooks' sensitive-column detection to join against `TAG_REFERENCES` instead —
  it will be materially more accurate in both directions.
- **Running classification for the first time is itself a good early-tenure deliverable.** If
  Section 4 comes back empty, proposing to run Snowflake's classification against the
  databases the heuristic notebooks already flagged as likely sensitive is a concrete,
  low-effort way to convert several notebooks' "probably sensitive" findings into confirmed
  ones.
- **Tags aren't only for sensitivity.** A low score on this scorecard doesn't necessarily mean
  the team is behind — some organizations tag for cost allocation only and handle data
  sensitivity a different way (dedicated schemas, naming conventions enforced by review). Ask
  before assuming tagging maturity is a gap rather than a deliberate choice.
